# Object-Oriented Programming (OOP) - Part 1
## Core Concepts: Inheritance and Encapsulation

Welcome to Day 1 of Object-Oriented Programming in Python! Today, we will dive deep into two fundamental pillars of OOP:
1. **Encapsulation**: Restricting direct access to some of an object's components, which is a mechanism of wrapping data (variables) and code acting on the data (methods) together as a single unit.
2. **Inheritance**: A mechanism that allows a new class (subclass/child class) to inherit attributes and methods from an existing class (superclass/parent class).

By the end of this notebook, you will:
* Understand access modifiers (Public, Protected, and Private) in Python.
* Know how Python implements **Name Mangling** under the hood.
* Use `@property` decorators to create getters, setters, and deleters with validation.
* Master Single, Multilevel, and Multiple Inheritance.
* Understand the Method Resolution Order (MRO) and the C3 Linearization algorithm.
* Build a comprehensive **Smart Home Device Ecosystem** capstone project.

---


## 1. Encapsulation: Controlling State & Access

Encapsulation is the practice of hiding the internal details of a class and exposing only what is necessary through a public interface. This protects the integrity of the data, prevents accidental modifications, and makes the code modular and easier to maintain.

### Access Modifiers in Python
Unlike languages like Java or C++, Python does not have strict keyword-based access modifiers (like `public`, `private`, `protected`). Instead, Python relies on naming conventions to signal the intended visibility of attributes and methods:

| Modifier | Naming Convention | Behavior / Intended Use |
|---|---|---|
| **Public** | `name` | Accessible from anywhere (inside or outside the class). |
| **Protected** | `_name` | Intended for internal use within the class and its subclasses. Python *does not* prevent external access, but developers treat it as a warning sign. |
| **Private** | `__name` | Intended to be completely hidden. Python triggers **Name Mangling** to make direct external access more difficult. |

### Name Mangling Under the Hood
When you prefix an attribute or method with double underscores (e.g., `__balance`), Python internally renames it to `_ClassName__attributeName`. This is called **Name Mangling**. It is not a security measure, but rather a protection mechanism to avoid name clashes in subclasses.


In [ ]:
# Let's explore access modifiers and name mangling in Python

class BankAccount:
    def __init__(self, owner: str, initial_balance: float):
        self.owner = owner                  # Public attribute
        self._routing_number = "121000248" # Protected attribute (convention only)
        self.__balance = initial_balance    # Private attribute (name mangling applied)
    
    # Public method
    def get_routing_info(self):
        return f"Routing Number: {self._routing_number}"
    
    # Private method
    def __reconcile(self):
        print("Performing internal reconciliation...")

# Instantiate a bank account
account = BankAccount("Alice", 1000.0)

print("--- Public Access ---")
print(f"Account Owner: {account.owner}")  # Works perfectly

print("\n--- Protected Access ---")
# Python allows this, but it is bad practice!
print(f"Routing Number: {account._routing_number}") 

print("\n--- Private Access (Fails Directly) ---")
try:
    print(account.__balance)
except AttributeError as e:
    print(f"AttributeError: {e}")

print("\n--- Name Mangling Unveiled ---")
# Let's inspect the object's dictionary to see how attributes are stored internally
print("Object attributes:", list(account.__dict__.keys()))

# Accessing the mangled name directly
print(f"Mangled Balance: {account._BankAccount__balance} (Success, but don't do this!)")

try:
    account.__reconcile()
except AttributeError as e:
    print(f"AttributeError for private method: {e}")

# Accessing mangled private method
account._BankAccount__reconcile()


## 2. Getters, Setters, and `@property`

Directly exposing variables (even public ones) can lead to invalid states. For instance, you wouldn't want a bank account balance to be set to a negative number or a text string. 

To safely access and modify attributes, we use **Getters** and **Setters**. In Python, the idiomatic way to implement them is using the `@property` decorator. 

### Why Use `@property`?
1. **Clean Syntax**: It allows you to access methods like attributes (e.g., `account.balance` instead of `account.get_balance()`).
2. **Encapsulated Validation**: You can intercept attribute assignment and validate input before saving it.
3. **Read-Only / Write-Only Attributes**: By omitting a setter, you can make attributes read-only.


In [ ]:
# Implementing validation and property decorators

class SmartThermostat:
    def __init__(self, location: str, target_temperature: float):
        self.location = location
        # Use private variable to hold the actual value
        self.__temperature = target_temperature
        
    # The Getter: exposes __temperature read access as a property
    @property
    def temperature(self) -> float:
        print("Getter called! Retrieving temperature...")
        return self.__temperature
    
    # The Setter: intercept writes and perform validation
    @temperature.setter
    def temperature(self, value: float) -> None:
        print(f"Setter called! Attempting to set temperature to {value}°C")
        if not isinstance(value, (int, float)):
            raise TypeError("Temperature must be a number.")
        if value < 10.0 or value > 35.0:
            raise ValueError("Temperature must be between 10.0°C and 35.0°C for safety.")
        self.__temperature = float(value)
        
    # The Deleter: defines behavior when 'del' is called on the property
    @temperature.deleter
    def temperature(self) -> None:
        print("Deleter called! Resetting temperature to default...")
        self.__temperature = 21.0  # Reset to default room temp

# Demo of property usage
thermostat = SmartThermostat("Living Room", 22.5)

print("--- Reading Temperature ---")
print(f"Current temperature setting: {thermostat.temperature}")

print("\n--- Modifying Temperature (Valid) ---")
thermostat.temperature = 24.0
print(f"New temperature setting: {thermostat.temperature}")

print("\n--- Modifying Temperature (Invalid Value) ---")
try:
    thermostat.temperature = 45.0
except ValueError as e:
    print(f"Validation Error: {e}")

print("\n--- Modifying Temperature (Invalid Type) ---")
try:
    thermostat.temperature = "Warm"
except TypeError as e:
    print(f"Type Error: {e}")

print("\n--- Deleting Temperature Property ---")
del thermostat.temperature
print(f"Temperature after deletion/reset: {thermostat.temperature}")


## 3. Inheritance: Reusing & Extending Code

Inheritance allows a new class (derived/child class) to inherit attributes and methods from an existing class (base/parent class). This avoids code duplication and enables **polymorphism** (the ability to treat different objects of different classes through a common interface).

### Key Concepts in Inheritance
1. **`super()`**: A built-in function that returns a proxy object that delegates method calls to a parent or sibling class. It is essential for calling parent constructors and executing parent methods in overridden submethods.
2. **Method Overriding**: Subclasses can define a method with the same name as a method in the parent class. The subclass's method will take precedence.

### Types of Inheritance
1. **Single Inheritance**: Child inherits from a single Parent.
2. **Multilevel Inheritance**: Grandchild inherits from Child, which inherits from Parent.
3. **Hierarchical Inheritance**: Multiple Children inherit from a single Parent.
4. **Multiple Inheritance**: A Child inherits from multiple independent Parents.
5. **Hybrid Inheritance**: A mix of two or more of the above types.


In [ ]:
# Demonstrating Single and Multilevel Inheritance

# Base class (Parent)
class Vehicle:
    def __init__(self, brand: str, model: str):
        self.brand = brand
        self.model = model
        self._engine_started = False
        
    def start_engine(self) -> None:
        self._engine_started = True
        print(f"The engine of {self.brand} {self.model} has started.")
        
    def stop_engine(self) -> None:
        self._engine_started = False
        print(f"The engine of {self.brand} {self.model} has stopped.")
        
    def __repr__(self) -> str:
        return f"Vehicle(brand={self.brand}, model={self.model})"

# Single Inheritance: Car inherits from Vehicle
class Car(Vehicle):
    def __init__(self, brand: str, model: str, doors: int):
        # Call the parent class's __init__ using super()
        super().__init__(brand, model)
        self.doors = doors
        
    # Method overriding: extend start_engine
    def start_engine(self) -> None:
        print("Checking seatbelts and electronic systems...")
        super().start_engine()  # Call the original base implementation
        print("Car is ready to drive!")

# Multilevel Inheritance: ElectricCar inherits from Car
class ElectricCar(Car):
    def __init__(self, brand: str, model: str, doors: int, battery_capacity: int):
        super().__init__(brand, model, doors)
        self.battery_capacity = battery_capacity # in kWh
        
    def charge_battery(self) -> None:
        print(f"Charging the {self.battery_capacity}kWh battery of the {self.brand} {self.model}...")

# Demo
print("--- Single Inheritance (Car) ---")
my_car = Car("Toyota", "Corolla", 4)
my_car.start_engine()
my_car.stop_engine()

print("\n--- Multilevel Inheritance (ElectricCar) ---")
my_tesla = ElectricCar("Tesla", "Model S", 4, 100)
my_tesla.start_engine()  # Inherits from Car (which overrides Vehicle)
my_tesla.charge_battery()  # Defined only in ElectricCar


## 4. Multiple Inheritance and Method Resolution Order (MRO)

Python supports **Multiple Inheritance**, which allows a class to inherit from more than one parent class. 

### The Diamond Problem
When Class B and Class C inherit from Class A, and Class D inherits from both B and C, which method does D execute if it calls a method defined in Class A that B and C override? This is the classic **Diamond Problem**.

```
    A
   / \
  B   C
   \ /
    D
```

### C3 Linearization and MRO
Python resolves this using the **C3 Linearization** algorithm, which generates a deterministic **Method Resolution Order (MRO)**. The MRO is the order in which Python looks for a method or attribute in a class hierarchy.

We can inspect the MRO of any class using:
* `ClassName.__mro__`
* `ClassName.mro()`
* The built-in `help(ClassName)` command


In [ ]:
# Demonstrating Multiple Inheritance, Mixins, and MRO

# Mixin Class 1: Provides connectivity features
class WiFiMixin:
    def __init__(self):
        self.connected_ssid = None
        
    def connect_to_wifi(self, ssid: str) -> None:
        self.connected_ssid = ssid
        print(f"Connected successfully to Wi-Fi network: '{ssid}'")
        
    def send_data(self, payload: dict) -> None:
        if self.connected_ssid:
            print(f"Sending payload over '{self.connected_ssid}': {payload}")
        else:
            print("Error: No Wi-Fi connection. Failed to send data.")

# Mixin Class 2: Provides logging features
class LoggingMixin:
    def __init__(self):
        self._logs = []
        
    def log_event(self, event: str) -> None:
        self._logs.append(event)
        print(f"[LOG] {event}")
        
    def print_logs(self) -> None:
        print("=== Event Logs ===")
        for log in self._logs:
            print(f" - {log}")

# Base Device Class
class Device:
    def __init__(self, serial_number: str):
        self.serial_number = serial_number
        self.power_on = False

    def toggle_power(self) -> None:
        self.power_on = not self.power_on
        state = "ON" if self.power_on else "OFF"
        print(f"Device {self.serial_number} powered {state}")

# SmartCamera: Inherits from Device and both Mixins
# Ordering matters: Python searches from left to right in the class definition
class SmartCamera(Device, WiFiMixin, LoggingMixin):
    def __init__(self, serial_number: str, resolution: str):
        # When using multiple inheritance, super() handles cooperative init.
        # However, because mixins might not call super().__init__() if they don't inherit from Device,
        # we explicitly initialize each parent class to ensure proper state setup.
        Device.__init__(self, serial_number)
        WiFiMixin.__init__(self)
        LoggingMixin.__init__(self)
        self.resolution = resolution
        
    def capture_image(self) -> None:
        self.log_event("Camera captured an image.")
        self.send_data({"event": "capture", "resolution": self.resolution})

# Instantiation
camera = SmartCamera("CAM-9988X", "4K Ultra HD")

print("--- Testing SmartCamera Features ---")
camera.toggle_power()
camera.connect_to_wifi("Home_Fiber_5G")
camera.capture_image()
camera.print_logs()

print("\n--- Method Resolution Order (MRO) ---")
# Let's inspect the order Python searches for methods:
for i, cls in enumerate(SmartCamera.mro(), 1):
    print(f"{i}. {cls.__name__}")


## 5. Capstone Project: Smart Home Hub System

Now that we have covered both Encapsulation and Inheritance in detail, let's put these concepts together to build a robust, real-world system: **A Smart Home Hub Controller**.

### Project Architecture & Requirements:
We will build a smart ecosystem where a central hub manages multiple home devices:
1. **Base Class `SmartDevice`**:
   - Encapsulation: Private serial number (`__serial_number`), protected status log (`_history_log`).
   - Use properties (`@property`) to expose and validate the power state and device name.
   - Core methods: `turn_on()`, `turn_off()`, and `log_action()`.
2. **Subclasses (Single & Multilevel Inheritance)**:
   - `SmartLight`: Inherits from `SmartDevice`, adds `brightness` control with property validations (0 to 100%). Overrides `__repr__` for elegant representation.
   - `SmartThermostat`: Inherits from `SmartDevice`, adds `temperature` control with property validation.
3. **Mixin Classes (Multiple Inheritance)**:
   - `RemoteAccessMixin`: Provides remote controlling functionality (token authentication).
   - `PowerSaverMixin`: Automatically dims lights or drops thermostat temperature when the device detects low activity.
4. **Hybrid Subclass**:
   - `AdvancedSmartLight`: Inherits from `SmartLight` and mixes in `RemoteAccessMixin` and `PowerSaverMixin`.
5. **Manager Class `SmartHub`**:
   - Keeps track of registered devices.
   - Demonstrates **polymorphism**: has a method `shutdown_all()` that turns off all devices regardless of their concrete class.
   - Has a method `run_diagnostics()` that lists status logs for all registered devices.


In [ ]:
import time
from typing import List, Dict, Any

# ==========================================
# 1. MIXINS (MULTIPLE INHERITANCE INGREDIENTS)
# ==========================================

class RemoteAccessMixin:
    """Provides remote management capabilities with simple authentication."""
    def __init__(self):
        self._auth_tokens = set()

    def register_remote_token(self, token: str) -> None:
        self._auth_tokens.add(token)
        print(f"[Remote Setup] Auth token '{token}' successfully registered.")

    def authenticate_remote_call(self, token: str) -> bool:
        return token in self._auth_tokens


class PowerSaverMixin:
    """Enables power-saving mode configurations on compatible devices."""
    def __init__(self):
        self._power_saver_enabled = False

    def toggle_power_saver(self) -> None:
        self._power_saver_enabled = not self._power_saver_enabled
        status = "ENABLED" if self._power_saver_enabled else "DISABLED"
        print(f"[Power Saver] Power-saving mode is now {status}.")

    @property
    def power_saver_enabled(self) -> bool:
        return self._power_saver_enabled


# ==========================================
# 2. BASE CLASS (WITH ENCAPSULATION & PROPERTIES)
# ==========================================

class SmartDevice:
    """Base class representing any smart household device."""
    def __init__(self, name: str, serial_number: str):
        self.name = name
        # Encapsulation: Private serial number (Name Mangling)
        self.__serial_number = serial_number
        # Encapsulation: Protected history log
        self._history_log: List[Dict[str, Any]] = []
        self._is_on = False
        self._log_event("Device initialized.")

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, new_name: str) -> None:
        if not new_name.strip():
            raise ValueError("Device name cannot be blank.")
        self._name = new_name

    @property
    def is_on(self) -> bool:
        return self._is_on

    # Encapsulated getter for private serial number (read-only)
    @property
    def serial_number(self) -> str:
        return self.__serial_number

    def _log_event(self, action: str) -> None:
        """Protected helper to record activity history."""
        self._history_log.append({
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "action": action
        })

    def turn_on(self) -> None:
        if not self._is_on:
            self._is_on = True
            self._log_event("Powered ON")
            print(f"[{self.name}] has been turned on.")

    def turn_off(self) -> None:
        if self._is_on:
            self._is_on = False
            self._log_event("Powered OFF")
            print(f"[{self.name}] has been turned off.")

    def get_logs(self) -> List[Dict[str, Any]]:
        """Public API to inspect device history logs."""
        return list(self._history_log)

    def __repr__(self) -> str:
        return f"<{self.__class__.__name__} name={self.name} on={self._is_on}>"


# ==========================================
# 3. DERIVED CLASSES (SINGLE / MULTILEVEL INHERITANCE)
# ==========================================

class SmartLight(SmartDevice):
    """A smart light bulb subclass with dimming capability."""
    def __init__(self, name: str, serial_number: str):
        super().__init__(name, serial_number)
        self._brightness = 100  # Default brightness in %

    @property
    def brightness(self) -> int:
        return self._brightness

    @brightness.setter
    def brightness(self, level: int) -> None:
        if not isinstance(level, int):
            raise TypeError("Brightness level must be an integer.")
        if level < 0 or level > 100:
            raise ValueError("Brightness must be between 0 and 100.")
        self._brightness = level
        self._log_event(f"Brightness adjusted to {level}%")
        print(f"[{self.name}] Brightness set to {level}%.")

    # Override turn_on to default to full brightness
    def turn_on(self) -> None:
        super().turn_on()
        self._brightness = 100

    def __repr__(self) -> str:
        return f"<SmartLight name={self.name} on={self.is_on} brightness={self._brightness}%>"


class SmartThermostat(SmartDevice):
    """A smart thermostat to regulate temperature."""
    def __init__(self, name: str, serial_number: str, default_temp: float = 21.0):
        super().__init__(name, serial_number)
        self.__temperature = default_temp

    @property
    def temperature(self) -> float:
        return self.__temperature

    @temperature.setter
    def temperature(self, target: float) -> None:
        if not isinstance(target, (int, float)):
            raise TypeError("Target temperature must be numerical.")
        if target < 15.0 or target > 30.0:
            raise ValueError("Target temperature must be within safe limits: 15.0°C to 30.0°C.")
        self.__temperature = float(target)
        self._log_event(f"Temperature set to {target}°C")
        print(f"[{self.name}] Temperature adjusted to {target}°C.")


# ==========================================
# 4. HYBRID CLASS (MULTIPLE INHERITANCE IN ACTION)
# ==========================================

class AdvancedSmartLight(SmartLight, RemoteAccessMixin, PowerSaverMixin):
    """A premium smart light equipped with power saving policies and remote security."""
    def __init__(self, name: str, serial_number: str):
        # Initialize parent classes
        SmartLight.__init__(self, name, serial_number)
        RemoteAccessMixin.__init__(self)
        PowerSaverMixin.__init__(self)
        self._log_event("Advanced SmartLight configured with Remote & PowerSaver features.")

    # Override property setter to apply power-saving logic
    @SmartLight.brightness.setter
    def brightness(self, level: int) -> None:
        if self.power_saver_enabled and level > 60:
            print(f"[Power Saver Block] Brightness capped at 60% on [{self.name}]. To go higher, disable Power Saver.")
            level = 60
        # Call the property setter of the superclass (SmartLight)
        # Using descriptor syntax since super() property routing can be complex
        SmartLight.brightness.fset(self, level)

    def remote_adjust_brightness(self, level: int, auth_token: str) -> None:
        """Remote access wrapper requiring authentication."""
        if self.authenticate_remote_call(auth_token):
            print(f"[Remote Access Approved] Request for {self.name} granted.")
            self.brightness = level
        else:
            print(f"[Remote Access Denied] Unauthorized access attempt detected on {self.name}!")
            self._log_event("Failed remote access attempt!")


# ==========================================
# 5. CONTAINER CLASS (DEMONSTRATING POLYMORPHISM)
# ==========================================

class SmartHomeHub:
    """A central coordinator that controls all devices in the ecosystem."""
    def __init__(self, hub_name: str):
        self.hub_name = hub_name
        self._devices: List[SmartDevice] = []

    def register_device(self, device: SmartDevice) -> None:
        if not isinstance(device, SmartDevice):
            raise TypeError("Only instances of SmartDevice subclasses can be registered.")
        self._devices.append(device)
        print(f"[Hub Registration] Registered '{device.name}' (Serial: {device.serial_number})")

    def shutdown_all(self) -> None:
        """Demonstrates polymorphism by calling turn_off on all registered devices."""
        print(f"\n--- Hub [{self.hub_name}]: Initiating Global Shutdown ---")
        for device in self._devices:
            # Polymorphic execution: each device type runs its respective turn_off implementation
            device.turn_off()

    def run_diagnostics(self) -> None:
        """Aggregates history logs from all registered devices."""
        print(f"\n--- Hub [{self.hub_name}]: System Diagnostics & Logs ---")
        for device in self._devices:
            print(f"Device: {device.name} | Type: {device.__class__.__name__}")
            for entry in device.get_logs():
                print(f"  [{entry['timestamp']}] -> {entry['action']}")


# ==========================================
# 6. RUNNING THE SYSTEM DEMO
# ==========================================

# 1. Instantiate the Hub
hub = SmartHomeHub("Central Nest")

# 2. Instantiate different smart devices
living_room_light = SmartLight("Living Room Light", "LT-001A")
hallway_light = AdvancedSmartLight("Hallway Security Light", "LT-999X")
bedroom_thermostat = SmartThermostat("Master Bedroom Thermostat", "TH-023B", default_temp=20.0)

# Register devices (Polymorphism: Hub treats all as SmartDevice instances)
hub.register_device(living_room_light)
hub.register_device(hallway_light)
hub.register_device(bedroom_thermostat)

# 3. Control devices locally
print("\n--- Initializing Device Controls ---")
living_room_light.turn_on()
living_room_light.brightness = 75

bedroom_thermostat.turn_on()
bedroom_thermostat.temperature = 23.5

# 4. Showcase Hybrid Class Features (Multiple Inheritance)
print("\n--- Testing Advanced Features on Hybrid Class ---")
hallway_light.turn_on()
hallway_light.register_remote_token("SECURE-KEY-456")

# Test remote control - authentication fail
hallway_light.remote_adjust_brightness(80, auth_token="WRONG-KEY")

# Test remote control - authentication success
hallway_light.remote_adjust_brightness(85, auth_token="SECURE-KEY-456")

# Enable power saving mode and test capping mechanism
hallway_light.toggle_power_saver()
# Try setting to 90% (should cap at 60% due to power saver)
hallway_light.brightness = 90
print(f"Current brightness: {hallway_light.brightness}%")

# 5. Global Shutdown and Diagnostics (Polymorphism)
hub.shutdown_all()
hub.run_diagnostics()


## 6. Student Exercise: Extend the Ecosystem

To reinforce your understanding of inheritance, encapsulation, and mixins, try completing the following challenge:

### Challenge Task:
Create a new mixin class called `WaterSensorMixin` and a derived device called `SmartSprinkler` that meets the following criteria:

1. **`WaterSensorMixin`**:
   - Private attribute `__water_usage_liters` (initialized to 0.0).
   - Public property `water_usage` to get the read-only current water usage.
   - Protected method `_consume_water(amount: float)` that increments the water usage.
2. **`SmartSprinkler`**:
   - Inherits from `SmartDevice` and mixes in `WaterSensorMixin`.
   - Has a public method `run_sprinkler(duration_minutes: float)`:
     - It should check if the sprinkler is powered ON first. If not, print a message saying it cannot run.
     - If it is ON, it should simulate water consumption. Assume a rate of **5.0 liters per minute**. Calculate the total consumed, call the protected `_consume_water` method to record it, and log the action `Sprinkler ran for X minutes. Consumed Y liters.`
3. **Registration & Test**:
   - Register a new `SmartSprinkler` instance with the `SmartHomeHub`.
   - Power it ON, run it for 10 minutes, and display the hub diagnostics to verify your sprinkler logs and water consumption.

Write your solution in the code cell below:


In [ ]:
# Write your solution here!
# 1. Define WaterSensorMixin
# 2. Define SmartSprinkler
# 3. Register and test with the SmartHomeHub
